Visualize prediction confidence using bar charts by manually computing probability distributions, implementing real-time updates with animations, and allowing users to interactively explore model confidence levels.

In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from matplotlib.animation import FuncAnimation

In [4]:
df = pd.read_csv("heart_disease_train.csv")

model = joblib.load("saved_models/heart_disease_model_v1.joblib")

print("Dataset loaded:", df.shape)
print("Model loaded successfully!")

Dataset loaded: (79, 19)
Model loaded successfully!


In [5]:
if hasattr(model, "feature_names_in_"):
    features = list(model.feature_names_in_)
else:
    with open("saved_models/heart_disease_model_v1_metadata.json") as f:
        metadata = json.load(f)

    features = metadata.get("features", metadata.get("feature_names"))

if not features:
    raise ValueError("Could not identify model features.")

classes = list(model.classes_)

print("Features:", features)
print("Prediction Classes:", classes)

Features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Prediction Classes: [np.int64(0), np.int64(1)]


In [6]:
def calculate_probabilities(input_data):

    if hasattr(model, "estimators_"):
        tree_probabilities = []

        for tree in model.estimators_:
            tree_probabilities.append(
                tree.predict_proba(input_data)[0]
            )

        probabilities = np.mean(tree_probabilities, axis=0)

    elif hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(input_data)[0]

    else:
        raise ValueError("Model does not support probability estimation.")

    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities / probabilities.sum()

    return probabilities

In [7]:
def calculate_probabilities(input_data):

    if hasattr(model, "estimators_"):
        tree_probabilities = []

        for tree in model.estimators_:
            tree_probabilities.append(
                tree.predict_proba(input_data)[0]
            )

        probabilities = np.mean(tree_probabilities, axis=0)

    elif hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(input_data)[0]

    else:
        raise ValueError("Model does not support probability estimation.")

    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities / probabilities.sum()

    return probabilities

In [8]:
input_widgets = {}

for feature in features:

    if pd.api.types.is_numeric_dtype(df[feature]):
        input_widgets[feature] = widgets.FloatText(
            value=float(df[feature].median()),
            description=feature,
            style={"description_width": "initial"}
        )

    else:
        categories = sorted(df[feature].dropna().unique().tolist())

        input_widgets[feature] = widgets.Dropdown(
            options=categories,
            value=categories[0],
            description=feature,
            style={"description_width": "initial"}
        )

display(widgets.VBox(list(input_widgets.values())))

In [9]:
def show_confidence_chart(probabilities):

    predicted_index = np.argmax(probabilities)

    fig, ax = plt.subplots(figsize=(8, 4))

    labels = [f"Class {c}" for c in classes]

    bars = ax.bar(
        labels,
        [0] * len(classes),
        width=0.5
    )

    ax.set_ylim(0, 1)
    ax.set_ylabel("Confidence")
    ax.set_title("Prediction Confidence Distribution")

    for i, probability in enumerate(probabilities):
        ax.text(i, probability + 0.02,
                f"{probability * 100:.2f}%",
                ha="center")

    def animate(frame):
        progress = min(frame / 20, 1)

        for i, bar in enumerate(bars):
            bar.set_height(probabilities[i] * progress)

        return bars

    animation = FuncAnimation(
        fig,
        animate,
        frames=21,
        interval=40,
        blit=False
    )

    plt.close(fig)

    display(HTML(animation.to_jshtml()))

In [10]:
output = widgets.Output()

def update_prediction(change=None):

    with output:
        clear_output(wait=True)

        user_data = {
            feature: input_widgets[feature].value
            for feature in features
        }

        input_df = pd.DataFrame([user_data], columns=features)

        try:
            probabilities = calculate_probabilities(input_df)

            prediction = model.predict(input_df)[0]

            print("Predicted Class:", prediction)
            print("Prediction Confidence:",
                  round(max(probabilities) * 100, 2), "%")

            print("\nClass Probability Distribution:")

            for cls, probability in zip(classes, probabilities):
                print(f"Class {cls}: {probability * 100:.2f}%")

            show_confidence_chart(probabilities)

        except Exception as e:
            print("Prediction Error:", e)


for widget in input_widgets.values():
    widget.observe(update_prediction, names="value")

display(output)

update_prediction()

Output()

In [11]:
probabilities = calculate_probabilities(
    pd.DataFrame(
        [{feature: input_widgets[feature].value for feature in features}]
    )
)

confidence_table = pd.DataFrame({
    "Class": classes,
    "Probability": probabilities,
    "Confidence (%)": probabilities * 100
})

display(confidence_table)

c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


,Class,Probability,Confidence (%)
0,0,0.236065,23.60655
1,1,0.763935,76.39345
